## Setup

Two-stage pipeline:
1. Run LPCMCI once and save raw graphs as JSON (`causal_graphs_raw/`).
2. Load those graphs, search discriminative paths, build rules, calibrate, and evaluate (`causal_path_search/`).

Re-run only the cell(s) whose stage you want to redo — the two stages are fully independent.

In [ ]:
import json
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

from charl_tre.causal.pipelines import PathSearchPipeline, RawDiscoveryPipeline
from charl_tre.settings import Settings

settings = Settings()

In [ ]:
# Stage 1: raw LPCMCI
IND_TEST = "parcorr"
TAU_MIN = 1
TAU_MAX = None  # None → settings.test_window_size // settings.train_window_size - 1
PC_ALPHA = 1e-3
STRIDE = None  # None → settings.train_window_size
USE_FULL_ONEHOT = True

# Stage 2: path search + classification
FILTER_THRESHOLD = 0.1
PATH_THRESHOLD = 0.0
MAX_EDGES_PER_PATH = 4
MIN_EDGES_PER_PATH = 2
TOP_PATHS_PER_ACTIVITY = 150
TOP_RULES_PER_ACTIVITY = 50
RULE_MIN_DELTA = 0.0
CLASSIFIER_STRIDE = 75
CLASSIFIER_TRAIN_RATIO = 0.7
SEGMENT_LENGTH = 64
SEGMENT_HOP = 12
MAX_SEGMENTS_PER_ACTIVITY = None

## Stage 1 — Causal Discovery with LPCMCI

In [ ]:
raw_result = RawDiscoveryPipeline(
    settings=settings,
    ind_test_name=IND_TEST,
    tau_min=TAU_MIN,
    tau_max=TAU_MAX,
    pc_alpha=PC_ALPHA,
    stride=STRIDE,
    use_full_onehot=USE_FULL_ONEHOT,
).run()

raw_dir = Path(raw_result["output_dir"])
print("Raw graphs saved to:", raw_dir)

## Stage 2 — Path Search and Deterministic Classification

In [ ]:
path_result = PathSearchPipeline(
    settings=settings,
    raw_dir=raw_dir,
    filter_threshold=FILTER_THRESHOLD,
    path_threshold=PATH_THRESHOLD,
    max_edges_per_path=MAX_EDGES_PER_PATH,
    min_edges_per_path=MIN_EDGES_PER_PATH,
    top_paths_per_activity=TOP_PATHS_PER_ACTIVITY,
    top_rules_per_activity=TOP_RULES_PER_ACTIVITY,
    rule_min_delta=RULE_MIN_DELTA,
    classifier_stride=CLASSIFIER_STRIDE,
    classifier_train_ratio=CLASSIFIER_TRAIN_RATIO,
    segment_length=SEGMENT_LENGTH,
    segment_hop=SEGMENT_HOP,
    max_segments_per_activity=MAX_SEGMENTS_PER_ACTIVITY,
).run()

output_dir = Path(path_result["output_dir"])
print("Path search outputs saved to:", output_dir)

## Inspect Results

In [ ]:
metrics = json.loads((output_dir / "deterministic_classifier_metrics.json").read_text())
summary = json.loads((output_dir / "graphs_summary.json").read_text())
rules   = json.loads((output_dir / "classification_rules.json").read_text())

print(f"n_latent_variables : {summary['n_latent_variables']}")
print(f"n_segments         : {metrics['n_segments']}")
print(f"accuracy           : {metrics['accuracy']:.3f}")
print(f"mean margin        : {metrics['mean_prediction_margin']:.3f}")

In [ ]:
# Top-3 rules per activity
for activity, activity_rules in rules["activities"].items():
    print(f"\n[{activity}] {len(activity_rules)} rules")
    for rule in activity_rules[:3]:
        print(" -", rule["rule_text"])

In [ ]:
# Per-activity edge counts and top path
for activity, info in summary["activities"].items():
    top_path = info["top_paths"][0]["path"] if info["top_paths"] else []
    print(f"[{activity}] edges={info['n_edges']}  top_path={' → '.join(top_path)}")

In [ ]:
# Confusion matrix (already saved as PNG; optionally display inline)

cm_path = output_dir / "deterministic_confusion_matrix.png"
if cm_path.exists():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(mpimg.imread(cm_path))
    ax.axis("off")
    plt.tight_layout()
    plt.show()